# Appendix

## First example

In [27]:
# Import the libraries
import torch.nn.functional as F
from torch.autograd import grad
import torch

In [28]:
# Intialize the tensors
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)


In [29]:
# Create the model
z = w1*x1 + b
a = torch.sigmoid(z)

In [30]:
# Calculate gradients individually
loss = F.binary_cross_entropy(a, y)
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [31]:
# Alternate calucation of the gradients
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## Regression from Scratch

In [32]:
import numpy as np 
from sklearn.linear_model import LinearRegression
import torch 
import torch.optim as optim
import torch.nn as nn 

In [33]:
# Generate data
true_b = 1
true_w = 2
N = 100

# Data generation
np.random.seed(42)
x = np.random.rand(N, 1)
epsilon = (.1 * np.random.randn(N, 1))
y = true_b + true_w*x + epsilon

In [34]:
# Do the test/train split
idx = np.arange(N)
np.random.shuffle(idx)

# Use the first 80 for train
train_idx = idx[:int(N*.8)]
val_idx = idx[int(N*.8):]

x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]


### Gradient Descent


In [35]:
# Step 0 - Initialize b and w
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)

print(b, w)

[0.49671415] [-0.1382643]


In [36]:
# Step 1 Compute predictions
yhat = b + w * x_train
yhat.size

80

In [37]:
# Compute the loss
error = (yhat - y_train)
loss = (error**2).mean()
loss

np.float64(2.7421577700550976)

In [38]:
# Step 3 - Compute Gradients
b_grad = 2 * error.mean()
w_grad = 2 * (x_train * error).mean()
print(b_grad, w_grad)

-3.044811379650508 -1.8337537171510832


In [39]:
# Step 4 - Update the parameters

# Sets the learning rate
lr = 0.1
print(b, w)

# Update the parameters
b = b - lr * b_grad 
w = w - lr * w_grad

print(b, w)

[0.49671415] [-0.1382643]
[0.80119529] [0.04511107]


### Gradient Descent in Numpy

In [40]:
# Step 0 - Initialize parameters
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)
print(b, w)

# Set the learning rate, eta
lr = 0.1

# Set the number of epochs
n_epochs = 1000

for epoch in range(n_epochs):
    # Compute the predicted output - Forward pass
    yhat = b + w * x_train
    
    # Compute loss
    error = (yhat - y_train)
    loss = (error**2).mean()
    
    # Compute gradients
    b_grad = 2 * error.mean()
    w_grad = 2 * (x_train * error).mean()
    
    # Update the parameters
    b = b - lr * b_grad
    w = w - lr * w_grad
    
print(b, w)
    


[0.49671415] [-0.1382643]
[1.02354094] [1.96896411]


### Gradient Descent with Pytorch

In [41]:
# Convert data to tensors
x_train_tensor = torch.as_tensor(x_train)
y_train_tensor = torch.as_tensor(y_train)


In [42]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [43]:
# Intialize x, y training tensors
x_train_tensor = torch.as_tensor(x_train).float().to(device)
y_train_tensor = torch.as_tensor(y_train).float().to(device)

In [44]:
torch.manual_seed(42)

# Initialize b and w
b = torch.randn(1, requires_grad=True, dtype=torch.float)
w = torch.randn(1, requires_grad=True, dtype=torch.float)

print(b, w)

tensor([0.3367], requires_grad=True) tensor([0.1288], requires_grad=True)


In [45]:
# Step 1 - Calculate the predictions: Forward Pass
yhat = b + w * x_train_tensor

# Step 2 - Compute loss
error = (yhat - y_train_tensor)
loss = (error**2).mean()

# Step 3 - Backward pass
loss.backward()

# Zero the gradients
b.grad.zero_(), w.grad.zero_()

print(b.grad, w.grad)

tensor([0.]) tensor([0.])


In [46]:
# Complete code

# Set Learning Rate
lr = 0.1

# Initialize parameters b and w
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float)
w = torch.randn(1, requires_grad=True, dtype=torch.float)

# Set the # of epochs
n_epochs = 1000

for epoch in range(n_epochs):
    # Compute the predictions - Forward pass
    yhat = b + w * x_train_tensor
    
    # Compute the loss
    error = (yhat - y_train_tensor)
    loss = (error**2).mean()
    
    # Compute the gradients
    loss.backward()
    
    # Update the parameters
    with torch.no_grad():
        b -= lr * b.grad
        w -= lr * w.grad
        
    # Let go of the grads
    b.grad.zero_()
    w.grad.zero_()
    
print(b, w)

tensor([1.0235], requires_grad=True) tensor([1.9690], requires_grad=True)


In [47]:
# With optimizer

# Set learning rate
lr = 0.1

# Step 0 - Initialize Parameters
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)

# Define the SGD optimizer to update parameters
optimizer = optim.SGD([b, w], lr=lr)

# Define the # of epochs
n_epochs = 1000

for epoch in range(n_epochs):
    # Compute predictions - forward pass
    yhat = b + w * x_train_tensor
    
    # Compute loss
    error = (yhat - y_train_tensor)
    loss = (error**2).mean()
    
    # Compute gradients
    loss.backward()
    
    # Update parmeters using gradients
    optimizer.step()
    
    # Zero gradient
    optimizer.zero_grad()
    
    
print(b, w)

tensor([1.0235], requires_grad=True) tensor([1.9690], requires_grad=True)


In [49]:
# Built in Loss Functions

# Set the learning rate
lr = 0.1

# Intialize the parameters
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)

# Initialize the optimizer
optimizer = optim.SGD([b, w], lr=lr)

# Define MSE loss
loss_fn = nn.MSELoss(reduction='mean')

# Set the number of epochs
n_epochs = 1000

for epoch in range(n_epochs):
    # Calc predicitons - Forward pass
    yhat = b + w * x_train_tensor
    
    # Calc loss
    loss = loss_fn(yhat, y_train_tensor)
    
    # Calc gradients
    loss.backward()
    
    # Adjust parameters
    optimizer.step()
    optimizer.zero_grad()
    
print(b, w)


tensor([1.0235], requires_grad=True) tensor([1.9690], requires_grad=True)
